# Notebook 3: Comparison — Exact vs C-Trapezoid vs Midpoint
C-Trapezoid **K** uses the corrected formula \((3a-3b+c)/\mathrm{den}\) (half of the previously published \(6a-6b+2c\) numerator).
Runs ALL THREE methods on the same data and compares side-by-side.

**Scatter figures (professor layout):** Midpoint and C-Trapezoid are **parallel** for the same quantity:
- `fig1_scatter_H_exact_vs_mid_and_ctrap_parallel.png` — \(H_{\mathrm{exact}}\) vs \(H_{\mathrm{mid}}\) | vs \(H_{\mathrm{ctrap}}\)
- `fig2_scatter_K_exact_vs_mid_and_ctrap_parallel.png` — same for \(K\)

Also: % error tables, depth plots, GPT2 / cross-family views.

In [ ]:
!pip install -q transformers torch safetensors pandas matplotlib seaborn

In [ ]:
import re, os, gc, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
OUT = 'outputs_compare'
os.makedirs(OUT, exist_ok=True)
print('Ready.')

In [ ]:
# ==============================================
# ALL THREE METHODS in one place
# ==============================================

def compute_octiles(w):
    return np.quantile(w, [1/8,2/8,3/8,4/8,5/8,6/8,7/8], method='linear')

def all_methods(w):
    """Compute exact, C-Trapezoid, and midpoint metrics for one weight array."""
    w = np.asarray(w, dtype=np.float64).ravel()
    O = compute_octiles(w)
    O1,O2,O3,O4,O5,O6,O7 = O
    a = O7-O1; b = O6-O2; c = O5-O3

    # ---------- EXACT (touch every point) ----------
    H_ex = np.mean(np.abs(w - O4))
    left = w[w <= O4]; right = w[w > O4]
    HL_ex = np.mean(np.abs(left - O2)) if len(left)>0 else np.nan
    HR_ex = np.mean(np.abs(right - O6)) if len(right)>0 else np.nan
    K_ex = (HL_ex + HR_ex)/(2*H_ex) if H_ex>0 else np.nan
    G_ex = (np.mean(w) - O4)/H_ex if H_ex>0 else np.nan

    # ---------- C-TRAPEZOID (7 octiles only) ----------
    Q0 = 4*O1-6*O2+4*O3-O4; Q1e = 4*O7-6*O6+4*O5-O4
    h = 1/8
    IL = h*(Q0+O1)/2+h*(O1+O2)/2+h*(O2+O3)/2+h*(O3+O4)/2
    IR = h*(O4+O5)/2+h*(O5+O6)/2+h*(O6+O7)/2+h*(O7+Q1e)/2
    H_ct = IR - IL
    den_ct = 3*a-2*b+3*c
    G_ct = (3*(O7+O1)-2*(O6+O2)+3*(O5+O3)-8*O4)/den_ct if den_ct!=0 else np.nan
    K_ct = (3*a-3*b+c)/den_ct if den_ct!=0 else np.nan  # corrected: was (6a-6b+2c)/den (2x too large)

    # ---------- MIDPOINT (crudest approximation) ----------
    H_mp = b/2
    G_mp = ((O6-O4)-(O4-O2))/b if b!=0 else np.nan
    K_mp = (a-c)/(2*b) if b!=0 else np.nan

    return {
        'O1':O1,'O2':O2,'O3':O3,'O4':O4,'O5':O5,'O6':O6,'O7':O7,
        'H_exact':H_ex, 'G_exact':G_ex, 'K_exact':K_ex,
        'H_L_exact':HL_ex, 'H_R_exact':HR_ex,
        'H_ctrap':H_ct, 'G_ctrap':G_ct, 'K_ctrap':K_ct,
        'H_mid':H_mp, 'G_mid':G_mp, 'K_mid':K_mp,
    }

print('All 3 methods defined.')

In [ ]:
# ==============================================
# CLASSIFIER
# ==============================================

def classify_param(name, family):
    if family == 'gpt2':
        m = re.search(r'h\.(\d+)\.', name)
        layer = int(m.group(1)) if m else -1
        if '.attn.c_attn.weight' in name:  return layer, 'attn_input'
        if '.attn.c_proj.weight' in name:  return layer, 'attn_output'
        if '.mlp.c_fc.weight' in name:     return layer, 'ffn_input'
        if '.mlp.c_proj.weight' in name:   return layer, 'ffn_output'
    elif family == 'opt':
        m = re.search(r'layers\.(\d+)\.', name)
        layer = int(m.group(1)) if m else -1
        if '.self_attn.q_proj.weight' in name: return layer, 'attn_input'
        if '.self_attn.k_proj.weight' in name: return layer, 'attn_input'
        if '.self_attn.v_proj.weight' in name: return layer, 'attn_input'
        if '.self_attn.out_proj.weight' in name: return layer, 'attn_output'
        if '.fc1.weight' in name: return layer, 'ffn_input'
        if '.fc2.weight' in name: return layer, 'ffn_output'
    elif family == 'pythia':
        m = re.search(r'layers\.(\d+)\.', name)
        layer = int(m.group(1)) if m else -1
        if '.attention.query_key_value.weight' in name: return layer, 'attn_input'
        if '.attention.dense.weight' in name: return layer, 'attn_output'
        if '.mlp.dense_h_to_4h.weight' in name: return layer, 'ffn_input'
        if '.mlp.dense_4h_to_h.weight' in name: return layer, 'ffn_output'
    return -1, None

In [ ]:
# ==============================================
# RUN ALL MODELS with ALL 3 METHODS
# ==============================================

MODELS = [
    ('openai-community/gpt2',        'gpt2',   'GPT2-Small'),
    ('openai-community/gpt2-medium', 'gpt2',   'GPT2-Medium'),
    ('facebook/opt-125m',            'opt',     'OPT-125M'),
    ('EleutherAI/pythia-160m',       'pythia',  'Pythia-160M'),
]

all_rows = []
for model_name, family, label in MODELS:
    from transformers import AutoModelForCausalLM
    print(f'\nLoading {model_name}...')
    model = AutoModelForCausalLM.from_pretrained(model_name); model.eval()
    for name, p in model.named_parameters():
        layer, mtype = classify_param(name, family)
        if mtype is None: continue
        w = p.detach().cpu().numpy().astype(np.float64).ravel()
        row = all_methods(w)
        row.update({'model':label,'param':name,'layer':layer,'type':mtype,'n':w.size})
        all_rows.append(row)
    del model; gc.collect()
    print(f'  Done: {label}')

df = pd.DataFrame(all_rows)
df.to_csv(f'{OUT}/all_compare.csv', index=False)
print(f'\nTotal: {len(df)} matrices with all 3 methods. Saved.')

In [ ]:
# ==============================================
# COMPUTE % ERRORS
# ==============================================

df['H_ct_err'] = ((df['H_ctrap'] - df['H_exact']) / df['H_exact'] * 100).round(2)
df['H_mp_err'] = ((df['H_mid']   - df['H_exact']) / df['H_exact'] * 100).round(2)
df['K_ct_err'] = ((df['K_ctrap'] - df['K_exact']) / df['K_exact'] * 100).round(2)
df['K_mp_err'] = ((df['K_mid']   - df['K_exact']) / df['K_exact'] * 100).round(2)

print('=== Average % error by model ===')
err = df.groupby('model')[['H_ct_err','H_mp_err','K_ct_err','K_mp_err']].mean().round(2)
display(err)
print('\nNegative = underestimates.  Closer to 0 = better.')

In [ ]:
# ==============================================
# FIG 1: SCATTER — H exact vs Midpoint | H exact vs C-Trapezoid (parallel)
# ==============================================

hmax = df[['H_exact', 'H_mid', 'H_ctrap']].max().max() * 1.05
hlim = [0.0, float(hmax)]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
panels = [
    ('H_mid', 'Midpoint', r'$H$ (Midpoint)'),
    ('H_ctrap', 'C-Trapezoid', r'$H$ (C-Trapezoid)'),
]
for ax, (ycol, name, ylab) in zip(axes, panels):
    for label in df['model'].unique():
        s = df[df['model'] == label]
        ax.scatter(s['H_exact'], s[ycol], s=15, alpha=0.6, label=label)
    ax.plot(hlim, hlim, 'k--', lw=0.8, alpha=0.45, zorder=0)
    ax.set_xlim(hlim)
    ax.set_ylim(hlim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel(r'$H_{\mathrm{exact}}$')
    ax.set_ylabel(ylab)
    ax.set_title(f'$H$: exact vs {name}')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.3)

fig.suptitle(r'MAD $H$: exact vs octile approximations (parallel)', fontsize=12, y=1.02)
plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.savefig(f'{OUT}/fig1_scatter_H_exact_vs_mid_and_ctrap_parallel.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved fig1: H parallel (Midpoint | C-Trapezoid). Dots on diagonal = good match.')

In [ ]:
# ==============================================
# FIG 2: SCATTER — K exact vs Midpoint | K exact vs C-Trapezoid (parallel)
# ==============================================

kcols = df[['K_exact', 'K_mid', 'K_ctrap']].replace([np.inf, -np.inf], np.nan).dropna(how='any')
kmin = max(0.0, float(kcols.min().min()) * 0.98)
kmax = float(kcols.max().max()) * 1.05
klim = [kmin, kmax]

fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
panels = [
    ('K_mid', 'Midpoint', r'$K$ (Midpoint)'),
    ('K_ctrap', 'C-Trapezoid', r'$K$ (C-Trapezoid)'),
]
for ax, (ycol, name, ylab) in zip(axes, panels):
    for label in df['model'].unique():
        s = df[df['model'] == label]
        ax.scatter(s['K_exact'], s[ycol], s=15, alpha=0.6, label=label)
    ax.plot(klim, klim, 'k--', lw=0.8, alpha=0.45, zorder=0)
    ax.set_xlim(klim)
    ax.set_ylim(klim)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel(r'$K_{\mathrm{exact}}$')
    ax.set_ylabel(ylab)
    ax.set_title(f'$K$: exact vs {name}')
    ax.legend(fontsize=7, loc='upper left')
    ax.grid(alpha=0.3)

fig.suptitle(r'MAD kurtosis $K$: exact vs octile approximations (parallel)', fontsize=12, y=1.02)
plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.savefig(f'{OUT}/fig2_scatter_K_exact_vs_mid_and_ctrap_parallel.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved fig2: K parallel (Midpoint | C-Trapezoid). Compare distance to diagonal across panels.')

In [ ]:
# ==============================================
# FIG 3: % ERROR HISTOGRAMS
# ==============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(df['H_ct_err'], bins=50, alpha=0.7, label='C-Trapezoid', color='steelblue')
ax.hist(df['H_mp_err'], bins=50, alpha=0.5, label='Midpoint', color='coral')
ax.axvline(0, color='black', lw=1)
ax.set_title('H: % error distribution'); ax.set_xlabel('% error from exact')
ax.legend()

ax = axes[1]
ax.hist(df['K_ct_err'], bins=50, alpha=0.7, label='C-Trapezoid', color='steelblue')
ax.hist(df['K_mp_err'], bins=50, alpha=0.5, label='Midpoint', color='coral')
ax.axvline(0, color='black', lw=1)
ax.set_title('K: % error distribution'); ax.set_xlabel('% error from exact')
ax.legend()

plt.tight_layout()
plt.savefig(f'{OUT}/fig3_error_histograms.png', dpi=200, bbox_inches='tight')
plt.show()
print('Blue (C-Trap) should be tighter around 0 than orange (Midpoint).')

In [ ]:
# ==============================================
# FIG 4: SIDE-BY-SIDE H vs depth — all 3 methods on same plot
# GPT2-Small only
# ==============================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('GPT2-Small: All 3 methods side by side', fontsize=13)

gpt2 = df[df['model']=='GPT2-Small']

for ax, ntype in zip(axes[0], ['attn_output', 'ffn_output']):
    s = gpt2[gpt2['type']==ntype].sort_values('layer')
    ax.plot(s['layer'], s['H_exact'], 'o-', label='H exact', color='black', lw=2)
    ax.plot(s['layer'], s['H_ctrap'], 's--', label='H C-Trap', color='steelblue')
    ax.plot(s['layer'], s['H_mid'], '^:', label='H Midpoint', color='coral')
    ax.set_title(f'{ntype}: H comparison'); ax.set_xlabel('Layer')
    ax.set_ylabel('H'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

for ax, ntype in zip(axes[1], ['attn_output', 'ffn_output']):
    s = gpt2[gpt2['type']==ntype].sort_values('layer')
    ax.plot(s['layer'], s['K_exact'], 'o-', label='K exact', color='black', lw=2)
    ax.plot(s['layer'], s['K_ctrap'], 's--', label='K C-Trap', color='steelblue')
    ax.plot(s['layer'], s['K_mid'], '^:', label='K Midpoint', color='coral')
    ax.set_title(f'{ntype}: K comparison'); ax.set_xlabel('Layer')
    ax.set_ylabel('K'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT}/fig4_gpt2_3methods.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 5: GPT2-Small vs GPT2-Medium (SAME FAMILY, DIFFERENT SCALE)
# ==============================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Same family scaling: GPT2-Small (12L) vs GPT2-Medium (24L)', fontsize=13)

for ax, ntype in zip(axes[0], ['attn_output', 'ffn_output']):
    for label, color in [('GPT2-Small','blue'),('GPT2-Medium','orange')]:
        s = df[(df['model']==label)&(df['type']==ntype)].sort_values('layer')
        mx = s['layer'].max()
        ax.plot(s['layer']/max(mx,1), s['H_ctrap'], 'o-', label=label, color=color, ms=4)
    ax.set_title(f'{ntype}: H (C-Trap)'); ax.set_xlabel('Normalised depth')
    ax.set_ylabel('H'); ax.legend(); ax.grid(alpha=0.3)

for ax, ntype in zip(axes[1], ['attn_output', 'ffn_output']):
    for label, color in [('GPT2-Small','blue'),('GPT2-Medium','orange')]:
        s = df[(df['model']==label)&(df['type']==ntype)].sort_values('layer')
        mx = s['layer'].max()
        ax.plot(s['layer']/max(mx,1), s['K_ctrap'], 'o-', label=label, color=color, ms=4)
    ax.set_title(f'{ntype}: K (C-Trap)'); ax.set_xlabel('Normalised depth')
    ax.set_ylabel('K'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT}/fig5_gpt2_scaling.png', dpi=200, bbox_inches='tight')
plt.show()
print('Question: does the depth pattern hold when you double the layers?')

In [ ]:
# ==============================================
# FIG 6: Cross-family — GPT2 vs OPT vs Pythia (SAME LAYERS)
# ==============================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Cross-family (all 12 layers): GPT2-Small vs OPT-125M vs Pythia-160M', fontsize=13)

cross = df[df['model'].isin(['GPT2-Small','OPT-125M','Pythia-160M'])]

for ax, ntype in zip(axes[0], ['attn_output', 'ffn_output']):
    for label in ['GPT2-Small','OPT-125M','Pythia-160M']:
        s = cross[(cross['model']==label)&(cross['type']==ntype)].sort_values('layer')
        ax.plot(s['layer'], s['H_ctrap'], 'o-', label=label, ms=4)
    ax.set_title(f'{ntype}: H (C-Trap)'); ax.set_xlabel('Layer')
    ax.set_ylabel('H'); ax.legend(); ax.grid(alpha=0.3)

for ax, ntype in zip(axes[1], ['attn_output', 'ffn_output']):
    for label in ['GPT2-Small','OPT-125M','Pythia-160M']:
        s = cross[(cross['model']==label)&(cross['type']==ntype)].sort_values('layer')
        ax.plot(s['layer'], s['K_ctrap'], 'o-', label=label, ms=4)
    ax.set_title(f'{ntype}: K (C-Trap)'); ax.set_xlabel('Layer')
    ax.set_ylabel('K'); ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT}/fig6_cross_family.png', dpi=200, bbox_inches='tight')
plt.show()
print('Question: same pattern across architectures or different?')

In [ ]:
# ==============================================
# FIG 7: Histograms — GPT2 attn_output at 4 depths
# ==============================================

from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained('openai-community/gpt2')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('GPT2-Small attn_output: weight shapes at layers 0, 4, 8, 11', fontsize=13)
for ax, lyr in zip(axes.flat, [0, 4, 8, 11]):
    pname = f'transformer.h.{lyr}.attn.c_proj.weight'
    w = dict(model.named_parameters())[pname].detach().cpu().numpy().ravel()
    lo, hi = np.percentile(w, [0.5, 99.5])
    ax.hist(w, bins=200, range=(lo,hi), density=True, color='steelblue', alpha=0.7, edgecolor='none')
    O = compute_octiles(w)
    for oi in O: ax.axvline(oi, color='coral', lw=0.8, alpha=0.6)
    ax.axvline(O[3], color='red', lw=1.5)
    m = all_methods(w)
    ax.set_title(f'Layer {lyr}: H_ex={m["H_exact"]:.4f} H_ct={m["H_ctrap"]:.4f} K_ex={m["K_exact"]:.3f} K_ct={m["K_ctrap"]:.3f}', fontsize=9)
    ax.set_xlabel('Weight value')

del model; gc.collect()
plt.tight_layout()
plt.savefig(f'{OUT}/fig7_histograms.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# TABLE: Full comparison for GPT2-Small
# ==============================================

gpt2 = df[df['model']=='GPT2-Small'][['layer','type',
    'H_exact','H_ctrap','H_mid','H_ct_err','H_mp_err',
    'K_exact','K_ctrap','K_mid','K_ct_err','K_mp_err']].sort_values(['type','layer'])
gpt2.to_csv(f'{OUT}/gpt2_small_comparison.csv', index=False)
print('=== GPT2-Small: Exact vs C-Trapezoid vs Midpoint ===')
with pd.option_context('display.max_rows', 60, 'display.float_format', '{:.4f}'.format):
    display(gpt2)

In [ ]:
# ==============================================
# TABLE: Summary % errors across all models
# ==============================================

print('=== AVERAGE % ERROR BY MODEL AND TYPE ===')
err_detail = df.groupby(['model','type'])[['H_ct_err','H_mp_err','K_ct_err','K_mp_err']].mean().round(2)
err_detail.to_csv(f'{OUT}/error_summary.csv')
display(err_detail)
print('\nC-Trap errors should be smaller than Midpoint errors.')

In [ ]:
# ==============================================
# FINAL SUMMARY
# ==============================================

print('='*60)
print('COMPARISON SUMMARY')
print('='*60)
print(f'\nH: C-Trap avg error = {df.H_ct_err.abs().mean():.2f}%,  Midpoint avg error = {df.H_mp_err.abs().mean():.2f}%')
print(f'K: C-Trap avg error = {df.K_ct_err.abs().mean():.2f}%,  Midpoint avg error = {df.K_mp_err.abs().mean():.2f}%')
print(f'\nK_exact range: [{df.K_exact.min():.3f}, {df.K_exact.max():.3f}]  (should be 0-1)')
print(f'K_ctrap range: [{df.K_ctrap.min():.3f}, {df.K_ctrap.max():.3f}]')
print(f'K_mid   range: [{df.K_mid.min():.3f}, {df.K_mid.max():.3f}]')

In [ ]:
try:
    from google.colab import files
    import glob
    for f in glob.glob(f'{OUT}/*'): files.download(f)
except: print('Files in:', OUT, os.listdir(OUT))